In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [3]:
df=pd.read_csv(r'C:\Users\Ajvad\Desktop\5th sem ojt\project 1_diabaties\data\processed\muhammed_ajvad_sinan_p1w2_features.csv')

In [4]:
df.head()

,patient_id,age,gender,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,...,smoker,last_checkup_date,outcome,age_band,bmi_category,glucose_band,insulin_glucose,pregnancies_rate,since_checkup,activity_smoker
0,P00826,79,male,0,124.0,71.0,30.0,124.0,23.4,0.588,...,NO,2023-04-20,0,Senior,Normal,Prediabetic,1.000000,0.000000,618,Unknown/NO
1,P00108,37,male,0,153.0,85.0,24.0,42.0,33.9,0.192,...,YES,2024-05-03,1,Middle-Aged,Obese,Diabetic,0.274510,0.000000,239,Medium/YES
2,P00517,39,female,0,142.0,68.0,22.0,159.0,26.9,0.777,...,UNKNOWN,2023-07-08,0,Middle-Aged,Over-weight,Diabetic,1.119718,0.000000,539,Low/UNKNOWN
3,P00687,68,female,7,121.0,88.0,26.0,124.0,29.0,1.217,...,YES,2024-04-14,1,Senior,Over-weight,Prediabetic,1.024793,0.102941,258,Unknown/YES
4,P00927,75,female,0,107.0,84.0,31.0,101.0,21.2,0.278,...,NO,2024-01-27,0,Senior,Normal,Prediabetic,0.943925,0.000000,336,Low/NO


In [5]:
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness','insulin','bmi','diabetes_pedigree','bmi_category','glucose_band']

x=df[feature_cols].copy()
y=df['outcome']
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi_category,glucose_band
0,79,0,124.000000,71.0,30.0,124.0,23.4,0.588,Normal,Prediabetic
1,37,0,153.000000,85.0,24.0,42.0,33.9,0.192,Obese,Diabetic
2,39,0,142.000000,68.0,22.0,159.0,26.9,0.777,Over-weight,Diabetic
3,68,7,121.000000,88.0,26.0,124.0,29.0,1.217,Over-weight,Prediabetic
4,75,0,107.000000,84.0,31.0,101.0,21.2,0.278,Normal,Prediabetic
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.855165,81.0,34.0,124.0,40.3,0.641,Obese,Prediabetic
946,71,0,140.000000,64.0,37.0,124.0,33.4,0.116,Obese,Diabetic
947,40,0,130.000000,73.0,26.0,258.0,20.1,0.709,Normal,Diabetic
948,52,4,123.000000,76.0,22.0,189.0,38.9,0.540,Obese,Prediabetic


In [6]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [7]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [8]:
print("train of x =",x_train.shape,"test",x_test.shape,)
print()
print("train of y =",y_train.shape,"test",y_test.shape)

train of x = (760, 13) test (190, 13)

train of y = (760,) test (190,)


In [9]:
print("y train",y_train.value_counts(normalize=True))
print()
print("y test",y_test.value_counts(normalize=True))

y train outcome
0    0.713158
1    0.286842
Name: proportion, dtype: float64

y test outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64


In [10]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)#Calculates the mean and standard deviation from x train
x_test_scaled=scaler.transform(x_test)#Uses those values to standardize x_train.s

In [11]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)

In [12]:
print('LogisticRegression',accuracy_score(y_test,logreg.predict(x_test_scaled)))
print('DecisionTreeClassifier',accuracy_score(y_test,dtree.predict(x_test)))
print('kneighborsclassifier',accuracy_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.7526315789473684
DecisionTreeClassifier 0.7210526315789474
kneighborsclassifier 0.7473684210526316


In [13]:
models = [
    ("Logistic Regression", logreg, x_test_scaled),
    ("Decision Tree (d=3)", dtree, x_test),
    ("KNN (k=25)", knn, x_test_scaled)
]

print("\nTest set:", len(y_test), "patients |",
      int(np.sum(y_test)), "of them diabetic")


Test set: 190 patients | 55 of them diabetic


In [14]:
rows = []

for name, m, Xt in models:
    cm = confusion_matrix(y_test, m.predict(Xt))
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "model": name,
        "correct_negatives": int(tn),
        "false_alarms": int(fp),
        "patients_missed": int(fn),
        "patients_found": int(tp),
        "accuracy": round(accuracy_score(y_test, m.predict(Xt)), 4)
    })

matrices = pd.DataFrame(rows)
print(matrices.to_string(index=False))

              model  correct_negatives  false_alarms  patients_missed  patients_found  accuracy
Logistic Regression                127             8               39              16    0.7526
Decision Tree (d=3)                121            14               39              16    0.7211
         KNN (k=25)                132             3               45              10    0.7474


* **127 (TN):** 127 people were correctly identified as not having diabetes.
* **8 (FP):** 8 people were incorrectly identified as having diabetes.
* **39 (FN):** 39 people who actually had diabetes were missed by the model.
* **16 (TP):** 16 people who had diabetes were correctly identified by the model.

* **Logistic Regression:** The main error is **missed patients (FN = 39)**.
* **Decision Tree:** The main error is **missed patients (FN = 39)**, along with 14 false alarms.
* **KNN:** The main error is **missed patients (FN = 45)**, which is the highest among the three models.

# score the model that never says yes

In [15]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_train,y_train)
y_pred = dummy.predict(x_test)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [16]:
print('DummyClassifier',accuracy_score(y_test,y_pred))

DummyClassifier 0.7105263157894737


In [17]:
y_test.value_counts(normalize=True)

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64

## precision and recall

RECALL

Recall asks: Of all the actual positives, how many did the model successfully find?

Recall = TP / (TP + FN)

In [18]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"recall ={(tp)}/{(tp)}+{(fn)}={tp / (tp + fn):.4f}")

recall =16/16+39=0.2909


import recall 

In [19]:
from sklearn.metrics import recall_score

In [20]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"recall ={(tp)}/{(tp)}+{(fn)}={tp / (tp + fn):.4f}")

print(f"recall_score={recall_score(y_test,logreg.predict(x_test_scaled)):.4f}")

recall =16/16+39=0.2909
recall_score=0.2909


In [21]:
print(f"Decision tree Recall Score = {recall_score(y_test, dtree.predict(x_test)):.4f}")

Decision tree Recall Score = 0.2909


In [22]:
print(f"KNN Recall Score = {recall_score(y_test, knn.predict(x_test_scaled)):.4f}")

KNN Recall Score = 0.1818


PRECISION

Precision asks: Of everything the model predicted as positive, how much was actually positive?

Precision = TP / (TP + FP)

In [23]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"precision ={(tp)}/{(tp)}+{(fp)}={tp / (tp + fp):.4f}")

precision =16/16+8=0.6667


import precision 

In [24]:
from sklearn.metrics import precision_score

In [25]:
cm = confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print(f"precision ={(tp)}/{(tp)}+{(fp)}={tp / (tp + fp):.4f}")

print(f"logistic regression precision_score={precision_score(y_test,logreg.predict(x_test_scaled)):.4f}")
print(f"Decision tree precision Score = {precision_score(y_test, dtree.predict(x_test)):.4f}")
print(f"KNN precision Score = {precision_score(y_test, knn.predict(x_test_scaled)):.4f}")

precision =16/16+8=0.6667
logistic regression precision_score=0.6667
Decision tree precision Score = 0.5333
KNN precision Score = 0.7692


## BUILD THE THREE METRUC TABLE

In [26]:
rows = []

for name, m, Xt in models:
    pred = m.predict(Xt)
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp),
        "flagged": int(tp+fp)
    })

result = pd.DataFrame(rows)


precision sort

In [27]:
print(result.sort_values(by="precision", ascending=False).to_string())

                 model  accuracy  precision  recall  found  missed  false_alarms  flagged
2           KNN (k=25)    0.7474     0.7692  0.1818     10      45             3       13
0  Logistic Regression    0.7526     0.6667  0.2909     16      39             8       24
1  Decision Tree (d=3)    0.7211     0.5333  0.2909     16      39            14       30


recall sort

In [28]:
print(result.sort_values(by="recall", ascending=False).to_string())

                 model  accuracy  precision  recall  found  missed  false_alarms  flagged
0  Logistic Regression    0.7526     0.6667  0.2909     16      39             8       24
1  Decision Tree (d=3)    0.7211     0.5333  0.2909     16      39            14       30
2           KNN (k=25)    0.7474     0.7692  0.1818     10      45             3       13


In [29]:
probs=logreg.predict_proba(x_test_scaled)[:,1]
preds=logreg.predict(x_test_scaled)

pd.DataFrame({"probability":probs[:10].round(3),
              "prediction":preds[:10],
              "actual":np.asarray(y_test)[:10]})

,probability,prediction,actual
0,0.289,0,0
1,0.032,0,0
2,0.182,0,0
3,0.182,0,0
4,0.177,0,1
5,0.161,0,0
6,0.419,0,0
7,0.473,0,1
8,0.286,0,0
9,0.121,0,0


In [30]:
sweep=[]
for t in [0.5,0.45,0.4,0.35,0.3,0.25,0.2,0.15]:
    pred_t=(probs>=t).astype(int)
    cm=confusion_matrix(y_test,pred_t)
    tn,fp,fn,tp=cm.ravel()

    sweep.append({
            "threshold":t,
            "accuracy": round(accuracy_score(y_test, pred_t), 4),
            "precision": round(precision_score(y_test, pred_t), 4),
            "recall": round(recall_score(y_test, pred_t), 4),
            "found": int(tp),
            "missed": int(fn),
            "false_alarms": int(fp),
            "flagged": int(tp+fp)
        })

sweep=pd.DataFrame(sweep)
sweep


,threshold,accuracy,precision,recall,found,missed,false_alarms,flagged
0,0.50,0.7526,0.6667,0.2909,16,39,8,24
1,0.45,0.7316,0.5556,0.3636,20,35,16,36
2,0.40,0.7105,0.5000,0.4364,24,31,24,48
3,0.35,0.7158,0.5082,0.5636,31,24,30,61
4,0.30,0.6947,0.4789,0.6182,34,21,37,71
5,0.25,0.6158,0.4043,0.6909,38,17,56,94
6,0.20,0.5789,0.3874,0.7818,43,12,68,111
7,0.15,0.5316,0.3692,0.8727,48,7,82,130


## F1 SCORE

F1 score is a metric that balances precision and recall to measure a model’s overall performance.

Equation:

F1=2×Precision×Recall /
Precision+Recall

	​


import f1 score


In [31]:
from sklearn.metrics import f1_score

In [32]:
print(f"f1_score_logistic regression={f1_score(y_test,logreg.predict(x_test_scaled)):.4f}"),
print(f"f1_score_decisiontree={f1_score(y_test,dtree.predict(x_test)):.4f}"),
print(f"f1_score_KNN={f1_score(y_test,knn.predict(x_test_scaled)):.4f}")

f1_score_logistic regression=0.4051
f1_score_decisiontree=0.3765
f1_score_KNN=0.2941


In [33]:
from sklearn.metrics import f1_score, classification_report
f1_score(y_test,pred)
print(classification_report(y_test,pred,digits=4))

              precision    recall  f1-score   support

           0     0.7458    0.9778    0.8462       135
           1     0.7692    0.1818    0.2941        55

    accuracy                         0.7474       190
   macro avg     0.7575    0.5798    0.5701       190
weighted avg     0.7526    0.7474    0.6864       190



In [34]:
result['f1']=[f1_score(y_test,m.predict(Xt))
              for model,m,Xt in models]
print(result[['model', 'accuracy', 'precision', 'recall', 'f1']].sort_values("f1", ascending=False))

                 model  accuracy  precision  recall        f1
0  Logistic Regression    0.7526     0.6667  0.2909  0.405063
1  Decision Tree (d=3)    0.7211     0.5333  0.2909  0.376471
2           KNN (k=25)    0.7474     0.7692  0.1818  0.294118


In [35]:
dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(x_train, y_train)

dummy_pred = dummy.predict(x_test)

print("Dummy Model Accuracy =", 
      accuracy_score(y_test, dummy_pred))

print("Dummy Model Precision =", 
      precision_score(y_test, dummy_pred, zero_division=0))

print("Dummy Model Recall =", 
      recall_score(y_test, dummy_pred, zero_division=0))

print("Dummy Model F1 Score =", 
      f1_score(y_test, dummy_pred, zero_division=0))

Dummy Model Accuracy = 0.7105263157894737
Dummy Model Precision = 0.0
Dummy Model Recall = 0.0
Dummy Model F1 Score = 0.0


find the threshold that maximise the f1

In [40]:
sweep['f1'] = [round(f1_score(y_test, (probs >= t).astype(int), zero_division=0),4)
               for t in sweep['threshold']]
print(sweep[['threshold', 'accuracy', 'precision', 'recall', 'f1', 'found']].sort_values("f1", ascending=False).to_string(index=False))


 threshold  accuracy  precision  recall     f1  found
      0.30    0.6947     0.4789  0.6182 0.5397     34
      0.35    0.7158     0.5082  0.5636 0.5345     31
      0.15    0.5316     0.3692  0.8727 0.5189     48
      0.20    0.5789     0.3874  0.7818 0.5181     43
      0.25    0.6158     0.4043  0.6909 0.5101     38
      0.40    0.7105     0.5000  0.4364 0.4660     24
      0.45    0.7316     0.5556  0.3636 0.4396     20
      0.50    0.7526     0.6667  0.2909 0.4051     16


In [ ]:
best = sweep.loc[sweep['f1'].idxmax()]  

Compare the tuned model with untuned one 

In [44]:
tuned_pred = (probs >= 0.30).astype(int)

def score_row(label, pred):
    cm = confusion_matrix(y_test,pred)
    tn,fp,fn,tp = cm.ravel()
    return{
        "model": label,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp)
    }

pd.DataFrame([
    score_row("Logistic Regression (threshold 0.50)", preds),
    score_row("Logistic Regression (threshold 0.30)", tuned_pred)])

,model,accuracy,precision,recall,f1,found,missed,false_alarms
0,Logistic Regression (threshold 0.50),0.7526,0.6667,0.2909,0.4051,16,39,8
1,Logistic Regression (threshold 0.30),0.6947,0.4789,0.6182,0.5397,34,21,37
